In [8]:
import pandas as pd
import numpy as np
from datasets import DatasetDict, Dataset
from sklearn.model_selection import train_test_split
import os


c:\Users\sukan\PycharmProjects\Polent_Extended\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
HF_DATA_FOLDER = "data/hf_format"
if not os.path.exists(HF_DATA_FOLDER):
    os.makedires(HF_DATA_FOLDER)

In [10]:
def summary_stats(dataset_dict, columns_to_value_count = None):
    split_df = {}
    for split, df in dataset_dict.items():
        split_df[split] = df.to_pandas()
    for split, df in split_df.items():
        print(f"Number of rows: {df.shape[0]}")
        print(f"Number of columns: {df.shape[1]}")
        print(f"Column names: {df.columns.tolist()}")
        print(f"Data types:")
        print(df.dtypes)
        print(f"Missing values:")
    if columns_to_value_count:
        for split, df in split_df.items():
            print(f"Value counts for {split}:")
            print(df[columns_to_value_count].value_counts())


In [11]:

def summary_stats(dataset_dict, columns_to_value_count=None):
    summary_rows = []
    value_count_rows = []

    for split, ds in dataset_dict.items():
        df = ds.to_pandas()  # assumes Hugging Face Dataset per split

        summary_rows.append(
            {
                "split": split,
                "num_items": df.shape[0],
                "num_columns": df.shape[1],
                "column_names": ", ".join(df.columns.astype(str).tolist()),
            }
        )

        if columns_to_value_count:
            for col in columns_to_value_count:
                if col not in df.columns:
                    continue

                counts = df[col].value_counts(dropna=False)
                for val, cnt in counts.items():
                    value_count_rows.append(
                        {
                            "split": split,
                            "column": col,
                            "value": val,
                            "count": int(cnt),
                            "percentage": f"{cnt/df.shape[0]:.2%}",
                        }
                    )

    summary_df = pd.DataFrame(summary_rows)
    value_counts_df = pd.DataFrame(value_count_rows)

    return summary_df, value_counts_df

## Park Data

The Park data is already divided in train, valid and test sets.
We rename the column `Park` as `label`. 

In [12]:
df_train = pd.read_csv("data/park/train.txt", encoding="utf-8", sep="\t", names = ["sent", "head", "tail", "Park"], header=None)
df_train.dropna(subset= ["Park","sent","head", "tail"],inplace=True)
df_train["Park"] = df_train["Park"].astype(int)
df_train.rename(columns={"Park": "labels"}, inplace=True)
df_train["head_tail"] = df_train.apply(lambda x: str(x["head"]) + " " + str(x["tail"]), axis = 1)
df_train.head()

,sent,head,tail,labels,head_tail
0,A [Los Angele]s news anchor said earlier this ...,Los Angele,the United State,0,Los Angele the United State
1,"President [Trump], who has often called for fe...",Trump,Kasich’,0,Trump Kasich’
2,"Cotton and other lawmakers, including Speaker ...",Paul Ryan,U. S.,0,Paul Ryan U. S.
3,[Congre]ss initially passed waivers to the ter...,Congre,Bush,0,Congre Bush
4,"Sorry Mr. Brooks, [Democrat]s aren't creating ...",Democrat,American,0,Democrat American


In [13]:
df_valid = pd.read_csv("data/park/valid.txt", encoding="utf-8", sep="\t", names = ["sent", "head", "tail", "Park"], header=None)
df_valid.dropna(subset= ["Park","sent","head", "tail"],inplace=True)
df_valid["Park"] = df_valid["Park"].astype(int)
df_valid.rename(columns={"Park": "labels"}, inplace=True)
df_valid["head_tail"] = df_valid.apply(lambda x: str(x["head"]) + " " + str(x["tail"]), axis = 1)
df_valid.head()

,sent,head,tail,labels,head_tail
0,[Kratz’]s claims would be a lot easier to beli...,Kratz’,Avery,0,Kratz’ Avery
1,[Brussel]s attacker [Najim Laachraoui].,Brussel,Najim Laachraoui,0,Brussel Najim Laachraoui
2,The [Conservative]s support is up from 32.4 pe...,Conservative,Liberal,0,Conservative Liberal
3,"The meeting, put together by Mr. Trump’s conse...",Paul Teller,Judicial Watch,0,Paul Teller Judicial Watch
4,The president's defeat highlighted the difficu...,Mubarak,Morsi,0,Mubarak Morsi


In [14]:
df_test = pd.read_csv("data/park/test.txt", encoding="utf-8", sep="\t", names = ["sent", "head", "tail", "Park"], header=None)
df_test.dropna(subset= ["Park","sent","head", "tail"],inplace=True)
df_test["Park"] = df_test["Park"].astype(int)
df_test.rename(columns={"Park": "labels"}, inplace=True)
df_test["head_tail"] = df_test.apply(lambda x: str(x["head"]) + " " + str(x["tail"]), axis = 1)
df_test.head()

,sent,head,tail,labels,head_tail
0,” [Breitbart New]s has now reached out to the ...,Breitbart New,Trump,0,Breitbart New Trump
1,"Phoebe Nora Mary Prince, 15, committed suicide...",Phoebe Prince,Mass.,0,Phoebe Prince Mass.
2,"[Notably], the department says about 30 emails...",Notably,Benghazi,0,Notably Benghazi
3,[Meacham] was recently featured by the San Die...,Meacham,Trump,0,Meacham Trump
4,But [Portland]'s city code prohibits persons f...,Portland,OregonLive.com,0,Portland OregonLive.com


In [15]:

data = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "valid": Dataset.from_pandas(df_valid),
    "test": Dataset.from_pandas(df_test)
})


summary_stats(dataset_dict = data, columns_to_value_count=["labels"])

(   split  num_items  num_columns                         column_names
 0  train      13144            5  sent, head, tail, labels, head_tail
 1  valid       1461            5  sent, head, tail, labels, head_tail
 2   test       1623            5  sent, head, tail, labels, head_tail,
     split  column  value  count percentage
 0   train  labels      0   8589     65.35%
 1   train  labels      3   2562     19.49%
 2   train  labels      1   1341     10.20%
 3   train  labels      4    387      2.94%
 4   train  labels      2    265      2.02%
 5   valid  labels      0    955     65.37%
 6   valid  labels      3    285     19.51%
 7   valid  labels      1    149     10.20%
 8   valid  labels      4     43      2.94%
 9   valid  labels      2     29      1.98%
 10   test  labels      0   1060     65.31%
 11   test  labels      3    316     19.47%
 12   test  labels      1    166     10.23%
 13   test  labels      4     48      2.96%
 14   test  labels      2     33      2.03%)

In [16]:
data.save_to_disk(os.path.join(HF_DATA_FOLDER, "park"))

Saving the dataset (1/1 shards): 100%|██████████| 1623/1623 [00:00<00:00, 367092.07 examples/s]


# TrainingData_ParkStyle

The Mueller report data which has been labelled in the Park Format

In [17]:
df = pd.read_csv("data/TrainingData_ParkStyle_2-0_forSukanya.csv")
df.dropna(subset= ["Park","sent","head", "tail"],inplace=True)
df["Park"] = df["Park"].astype(int)
df.rename(columns={"Park": "labels"}, inplace=True)
df["head_tail"] = df.apply(lambda x: str(x["head"]) + " " + str(x["tail"]), axis = 1)
df.head()

,sentID,sent,head,tail,labels,head_tail
0,4,"Though there has been ""forcefulness"" on both [...",Donald Trump,Newsom,1,Donald Trump Newsom
1,5,Cognizant of balancing [Trump]'s penchant for ...,Trump,Newsom,1,Trump Newsom
2,6,"At this delicate diplomatic moment, [Newsom] h...",Newsom,Trump,1,Newsom Trump
3,7,"""We are clearly operating under a different se...",Newsom,Trump,3,Newsom Trump
4,12,Gov. [Gavin Newsom] looks on during a news con...,Gavin Newsom,Xavier Becerra,0,Gavin Newsom Xavier Becerra


In [18]:
df_train, temp_df = train_test_split(df, test_size=0.30, stratify=df["labels"], random_state=42)
# Trying to ensure that all splits have all the classes
df_valid, df_test = train_test_split(temp_df, test_size=0.50, stratify=temp_df["labels"], random_state=42)
data = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "valid": Dataset.from_pandas(df_valid),
    "test": Dataset.from_pandas(df_test)
})

In [19]:
summary_stats(dataset_dict = data, columns_to_value_count=["labels"])

(   split  num_items  num_columns  \
 0  train       4520            7   
 1  valid        969            7   
 2   test        969            7   
 
                                         column_names  
 0  sentID, sent, head, tail, labels, head_tail, _...  
 1  sentID, sent, head, tail, labels, head_tail, _...  
 2  sentID, sent, head, tail, labels, head_tail, _...  ,
     split  column  value  count percentage
 0   train  labels      0   2026     44.82%
 1   train  labels      1   1123     24.85%
 2   train  labels      2    650     14.38%
 3   train  labels      3    552     12.21%
 4   train  labels      4    169      3.74%
 5   valid  labels      0    434     44.79%
 6   valid  labels      1    241     24.87%
 7   valid  labels      2    139     14.34%
 8   valid  labels      3    118     12.18%
 9   valid  labels      4     37      3.82%
 10   test  labels      0    434     44.79%
 11   test  labels      1    241     24.87%
 12   test  labels      2    140     14.45%
 13   tes

In [20]:
data.save_to_disk(os.path.join(HF_DATA_FOLDER, "polent_data_park_style"))

Saving the dataset (1/1 shards): 100%|██████████| 969/969 [00:00<?, ? examples/s]


# NYT articles for Trump and Musk

The NYT articles for Trump and Musk data. Note that this dataset has no labels and is meant to be predicted by other trained models. 

The summary is generated for completeness but without any labels.

In [21]:
df = pd.read_csv("data/nyt_articles_trump_musk.csv")
df.head()

,title,date,sentence,head,tail
0,Can Musk Pull Trump Back From the Tariff Ledge?,2025-04-07,Elon Musk has been criticizing Trump’s tariff ...,Trump,Elon Musk
1,Can Musk Pull Trump Back From the Tariff Ledge?,2025-04-07,Musk was expressing his discomfort with Presid...,Trump,Musk
2,Can Musk Pull Trump Back From the Tariff Ledge?,2025-04-07,And Musk repeatedly went out of his way to dis...,Trump,Musk
3,Can Musk Pull Trump Back From the Tariff Ledge?,2025-04-07,This is not the first time Musk has been at od...,Trump,Musk
4,Can Musk Pull Trump Back From the Tariff Ledge?,2025-04-07,"During the transition, Musk endorsed Rick Scot...",Trump,Musk


In [22]:
df_train, temp_df = train_test_split(df, test_size=0.30, random_state=42)
df_valid, df_test = train_test_split(temp_df, test_size=0.50, random_state=42)
data = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "valid": Dataset.from_pandas(df_valid),
    "test": Dataset.from_pandas(df_test)
})
summary_stats(dataset_dict = data)

(   split  num_items  num_columns  \
 0  train        259            6   
 1  valid         55            6   
 2   test         56            6   
 
                                         column_names  
 0  title, date, sentence, head, tail, __index_lev...  
 1  title, date, sentence, head, tail, __index_lev...  
 2  title, date, sentence, head, tail, __index_lev...  ,
 Empty DataFrame
 Columns: []
 Index: [])